# BIDS-like Folder Convention for SimNIBS

**Learning goal:** show *why* a strict naming convention makes methodological checks
nearly free — whereas an unstructured folder layout makes them tedious and error-prone.

> ⚠️ **Honest disclaimer**
> - There is **no** official BIDS standard for SimNIBS, and no `simnibs-bids` package exists.
> - The convention below is a **homemade proposal** inspired by real BIDS rules:
>   derivatives live in `derivatives/<pipeline>/`, entities are `key-value` pairs,
>   and `desc-<label>` is used to disambiguate when needed.
> - The `snb.query(...)` sketched here is implemented in **~30 lines of pure Python**:
>   it is an illustrative prototype, not an installable dependency.

## 1. The proposed convention

```
study/
└── derivatives/
    ├── simnibs-preparation/                # ① per-subject head model + masks
    │   ├── dataset_description.json
    │   └── sub-01/
    │       ├── task-charm/                 # CHARM output (head mesh)
    │       ├── task-charm_desc-T1only/     # variant: T1 without T2
    │       ├── mask-lM1_space-T1w/         # custom ROI mask, native space
    │       ├── mask-lM1_space-MNI/         # same mask, MNI space
    │       └── mask-rM1_space-T1w/
    ├── simnibs-simulation/                 # ② forward simulations
    │   ├── dataset_description.json
    │   └── sub-01/
    │       ├── montage-M1SO_roi-lM1/        # 1 condition = 1 folder
    │       ├── montage-M1SO_roi-rM1/
    │       └── montage-bil_roi-lM1/
    └── simnibs-optimization/               # ③ optimisation runs
        ├── dataset_description.json
        └── sub-01/
            └── target-lM1_constr-tot2mA_opt-tes/
```

**Pipeline dependency:** `preparation → simulation → optimization`
A subject should never appear in `simnibs-simulation/` without a corresponding
`task-charm/` folder in `simnibs-preparation/` (Check D below enforces this).

### Entities (all in `key-value` format)

| Entity          | Pipeline(s)          | Meaning                                    | Example              |
|-----------------|----------------------|--------------------------------------------|----------------------|
| `sub-<label>`   | all                  | subject / head model                       | `sub-01`             |
| `ses-<label>`   | all                  | session (optional)                         | `ses-pre`            |
| `task-<label>`  | preparation          | preparation step                           | `task-charm`         |
| `mask-<label>`  | preparation          | ROI mask name                              | `mask-lM1`           |
| `space-<label>` | preparation          | coordinate space                           | `space-T1w`          |
| `atlas-<label>` | preparation          | source atlas for mask (optional)           | `atlas-HCP`          |
| `montage-<lbl>` | simulation           | electrode montage / coil configuration     | `montage-M1SO`       |
| `roi-<lbl>`     | simulation           | region of interest                         | `roi-lM1`            |
| `target-<lbl>`  | optimization         | optimisation target                        | `target-lM1`         |
| `constr-<lbl>`  | optimization         | constraint                                 | `constr-tot2mA`      |
| `opt-<lbl>`     | optimization         | optimiser / modality                       | `opt-tes`            |
| `desc-<lbl>`    | all                  | free disambiguation (standard BIDS entity) | `desc-T1only`        |

**Golden rules:**
1. One experimental condition ⇄ exactly one folder.
2. No spaces; `_` separates entities, `-` separates key from value.
3. The *type* (`preparation` / `simulation` / `optimization`) lives in the **pipeline folder**, not as a prefix.
4. Entity labels must be **alphanumeric** — no `_` or `-` inside a value.

## 2. Build a demo directory tree

We create a fake dataset in a temporary folder, with a few intentional gaps
(missing conditions) so we can detect them later.

In [ ]:
import tempfile, os
from pathlib import Path

ROOT = Path(tempfile.mkdtemp()) / "study"

# (pipeline, sub, condition-folder)
conditions = [
    # --- preparation: CHARM head models ---
    ("simnibs-preparation", "sub-01", "task-charm"),
    ("simnibs-preparation", "sub-01", "mask-lM1_space-T1w"),
    ("simnibs-preparation", "sub-01", "mask-rM1_space-T1w"),
    ("simnibs-preparation", "sub-01", "mask-lM1_space-MNI"),
    ("simnibs-preparation", "sub-02", "task-charm"),
    ("simnibs-preparation", "sub-02", "mask-lM1_space-T1w"),
    # sub-02: mask-rM1 is MISSING -> intentional gap
    # sub-03: no preparation at all -> will be caught by Check D
    # --- simulations ---
    ("simnibs-simulation", "sub-01", "montage-M1SO_roi-lM1"),
    ("simnibs-simulation", "sub-01", "montage-M1SO_roi-rM1"),
    ("simnibs-simulation", "sub-01", "montage-bil_roi-lM1"),
    ("simnibs-simulation", "sub-02", "montage-M1SO_roi-lM1"),
    # sub-02: montage-M1SO_roi-rM1 is MISSING -> intentional gap
    ("simnibs-simulation", "sub-02", "montage-bil_roi-lM1"),
    ("simnibs-simulation", "sub-03", "montage-M1SO_roi-lM1"),  # no charm! -> Check D
    # --- optimisations ---
    ("simnibs-optimization", "sub-01", "target-lM1_constr-tot2mA_opt-tes"),
    ("simnibs-optimization", "sub-02", "target-lM1_constr-tot2mA_opt-tes"),
]

for pipeline, sub, cond in conditions:
    d = ROOT / "derivatives" / pipeline / sub / cond
    d.mkdir(parents=True, exist_ok=True)
    (d / "subject_TDCS_1_scalar.msh").write_text("fake mesh")

print("Dataset created at:", ROOT)


In [ ]:
# Quick tree visualisation
def show_tree(base: Path, prefix=""):
    entries = sorted(base.iterdir())
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + p.name)
        if p.is_dir():
            show_tree(p, prefix + ("    " if last else "│   "))

show_tree(ROOT)


## 3. Parse folder names → entity dictionaries

The core insight: because the name encodes everything, we can **reconstruct
all metadata without opening a single file** — exactly the logic behind `pybids`.

In [ ]:
def parse_entities(folder_name: str) -> dict:
    # 'montage-M1SO_roi-lM1' -> {'montage': 'M1SO', 'roi': 'lM1'}
    out = {}
    for token in folder_name.split("_"):
        if "-" in token:
            key, val = token.split("-", 1)
            out[key] = val
    return out

# quick sanity check
print(parse_entities("montage-M1SO_roi-lM1"))
print(parse_entities("target-lM1_constr-tot2mA_opt-tes"))


## 4. A minimal `snb.query(...)` (homemade prototype)

We reproduce the idea from your sketch — `snb.query(by_target, by_modality)` —
by scanning `derivatives/`, parsing each condition folder, and filtering on demand.

In [ ]:
class SimnibsBidsLayout:
    # Minimal pybids-style prototype — NOT a real package.
    def __init__(self, root):
        self.root = Path(root)
        self.records = list(self._index())

    def _index(self):
        deriv = self.root / "derivatives"
        for pipeline_dir in sorted(deriv.iterdir()):
            if not pipeline_dir.is_dir():
                continue
            for sub_dir in sorted(pipeline_dir.iterdir()):
                if not sub_dir.name.startswith("sub-"):
                    continue
                for cond in sorted(sub_dir.iterdir()):
                    if not cond.is_dir():
                        continue
                    ent = parse_entities(cond.name)
                    ent["pipeline"] = pipeline_dir.name
                    ent["sub"] = sub_dir.name.split("-", 1)[1]
                    ent["path"] = cond
                    yield ent

    def query(self, **filters):
        # e.g. query(roi='lM1', pipeline='simnibs-simulation')
        return [r for r in self.records
                if all(r.get(k) == v for k, v in filters.items())]

layout = SimnibsBidsLayout(ROOT)
print(f"{len(layout.records)} conditions indexed\n")

# --- query 1: all simulations targeting ROI lM1 ---
print("== Simulations targeting roi=lM1 ==")
for r in layout.query(pipeline="simnibs-simulation", roi="lM1"):
    print(" ", r["sub"], "|", r["path"].name)

# --- query 2: by optimisation modality ---
print("\n== Optimisations with opt=tes ==")
for r in layout.query(pipeline="simnibs-optimization", opt="tes"):
    print(" ", r["sub"], "|", r["path"].name)

# --- query 3: all masks available in MNI space ---
print("\n== Masks in MNI space ==")
for r in layout.query(pipeline="simnibs-preparation", space="MNI"):
    print(" ", r["sub"], "|", r["path"].name)


## 5. The real payoff: **methodological checks become trivial**

Because everything is queryable, we can automatically verify the **completeness**
and **consistency** of our experimental design — instead of clicking through folders
and hoping we haven't missed anything.

### Check A — Completeness matrix (subjects × conditions)
*"Did I actually run every condition for every subject?"*

In [ ]:
import itertools

# expected design (the theoretical plan)
expected_montages = ["M1SO", "bil"]
expected_rois     = ["lM1", "rM1"]
subjects          = ["01", "02"]

sims = layout.query(pipeline="simnibs-simulation")
present = {(r["sub"], r.get("montage"), r.get("roi")) for r in sims}

print(f"{'sub':<6}{'montage':<10}{'roi':<6}{'status'}")
print("-" * 30)
missing = []
for sub, mont, roi in itertools.product(subjects, expected_montages, expected_rois):
    ok = (sub, mont, roi) in present
    status = "✓" if ok else "✗ MISSING"
    if not ok:
        missing.append((sub, mont, roi))
    print(f"{sub:<6}{mont:<10}{roi:<6}{status}")

print(f"\n>>> {len(missing)} missing condition(s): {missing}")


### Check B — Cross-subject consistency
*"Do all subjects have exactly the same set of conditions?"*
(Catches subjects processed differently — a classic methodological mistake.)

In [ ]:
from collections import defaultdict

by_sub = defaultdict(set)
for r in layout.query(pipeline="simnibs-simulation"):
    by_sub[r["sub"]].add(r["path"].name)

all_conditions = set().union(*by_sub.values())
print("All conditions seen across subjects:")
for c in sorted(all_conditions):
    print("  -", c)

print("\nPer-subject differences:")
for sub, conds in sorted(by_sub.items()):
    diff = all_conditions - conds
    if diff:
        print(f"  sub-{sub}: MISSING {sorted(diff)}")
    else:
        print(f"  sub-{sub}: complete ✓")


### Check C — Duplicate / collision detection
If two runs produce the **same** condition folder name, it means either a silent
overwrite or a naming ambiguity. With the convention, this is a one-liner.

In [ ]:
from collections import Counter

# uniqueness key = (pipeline, sub, sorted entity set)
keys = []
for r in layout.records:
    ent = {k: v for k, v in r.items() if k != "path"}
    keys.append((ent["pipeline"], ent["sub"],
                 tuple(sorted((k, v) for k, v in ent.items()
                              if k not in ("pipeline", "sub")))))

dups = [k for k, n in Counter(keys).items() if n > 1]
print("Duplicates detected:", dups if dups else "none ✓")


### Check D — Preparation prerequisites
*"Does every subject with simulations have a completed CHARM head model?"*
This is the cross-pipeline dependency check — the most important one to run
before any batch of simulations.

In [ ]:
# subjects with at least one simulation
subs_with_simu = {r["sub"] for r in layout.query(pipeline="simnibs-simulation")}

# subjects with a completed CHARM preparation
subs_with_charm = {r["sub"] for r in layout.query(pipeline="simnibs-preparation", task="charm")}

print(f"Subjects with simulations : {sorted(subs_with_simu)}")
print(f"Subjects with CHARM model : {sorted(subs_with_charm)}")

missing_charm = subs_with_simu - subs_with_charm
if missing_charm:
    print(f"\n⚠️  MISSING CHARM for: {sorted(missing_charm)}")
    print("   → These subjects have simulations but no head model. Results are invalid.")
else:
    print("\n✓  All subjects with simulations have a CHARM model.")


## 6. Summary

What the convention buys you, concretely:

- **Queryability** — find "all simulations targeting lM1" without opening a single file.
- **Automatic methodological checks** — completeness matrix, cross-subject consistency,
  duplicate detection, **and cross-pipeline prerequisites** (Cells 5A / B / C / D).
- **Reproducibility** — the folder name *is* the metadata; nothing gets lost.
- **Extensibility** — adding an entity (`ses-`, `desc-`) doesn't break existing structure.

### Caveats
- This is **not** an official standard: document it in a `dataset_description.json`
  and a lab README so collaborators know the rules.
- The `SimnibsBidsLayout` prototype is minimal — for a production setup, check whether
  **`pybids`** (a real, maintained package) can index your structure; it was designed
  for raw BIDS data but may tolerate custom derivative entities.
- Entity labels must remain **alphanumeric** (no `_` or `-` inside a value),
  otherwise the parser breaks — this is also an explicit BIDS rule.